Import libraries

In [2]:
import numpy as np
import pandas as pd

Read dataset

In [4]:
df = pd.read_csv("C:/Users/dania/Documents/GIT/real-estate-data-platform/data/raw/realtor-data.csv")

## 1. Dataset Overview

This dataset contains real estate listings from the United States, including information about property prices, location, size, bedrooms, bathrooms, and previous sales history.

The objective of this analysis is to understand the structure, quality, and characteristics of the dataset before designing the data model and ETL pipeline.

Initial exploration

In [5]:
df.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


In [6]:
df.shape

(2226382, 12)

In [32]:
df.duplicated().sum()

np.int64(0)

## 2. Dataset Structure

Review column names, data types and memory usage to identify potential transformations required before loading the data into PostgreSQL.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   brokered_by     float64
 1   status          object 
 2   price           float64
 3   bed             float64
 4   bath            float64
 5   acre_lot        float64
 6   street          float64
 7   city            object 
 8   state           object 
 9   zip_code        float64
 10  house_size      float64
 11  prev_sold_date  object 
dtypes: float64(8), object(4)
memory usage: 203.8+ MB


## 3. Missing Values Analysis

This section evaluates the completeness of the dataset by identifying missing values and calculating their percentage across all columns.

Understanding missing data is essential before applying cleaning and transformation rules.

In [8]:
df.isnull().sum()

brokered_by         4533
status                 0
price               1541
bed               481317
bath              511771
acre_lot          325589
street             10866
city                1407
state                  8
zip_code             299
house_size        568484
prev_sold_date    734297
dtype: int64

In [26]:
(
    df.isnull().sum()
    / len(df)
    * 100
).round(2)

brokered_by        0.20
status             0.00
price              0.07
bed               21.62
bath              22.99
acre_lot          14.62
street             0.49
city               0.06
state              0.00
zip_code           0.01
house_size        25.53
prev_sold_date    32.98
dtype: float64

### Findings

- State, city, zip code and price contain very few missing values.
- Bedrooms, bathrooms and house size contain a significant number of missing values.
- Previous sale date has the highest percentage of missing values.

In [28]:
df[df['bed'].isnull()]['status'].value_counts()

status
for_sale          411341
sold               69974
ready_to_build         2
Name: count, dtype: int64

In [29]:
df[df['house_size'].isnull()]['status'].value_counts()

status
for_sale          464763
sold              103710
ready_to_build        11
Name: count, dtype: int64

In [30]:
df[df['bed'].isnull()].head(20)

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
42,52707.0,for_sale,400000.0,NaN,NaN,0.99,1857015.0,Moca,Puerto Rico,676.0,43082.0,NaN
43,80224.0,for_sale,380000.0,NaN,NaN,18.45,1855421.0,Hatillo,Puerto Rico,659.0,NaN,NaN
59,46019.0,for_sale,35000.0,NaN,NaN,0.88,1234597.0,Aguada,Puerto Rico,602.0,NaN,NaN
60,52464.0,for_sale,58000.0,NaN,NaN,0.21,1966337.0,Aguada,Puerto Rico,602.0,NaN,NaN
61,46019.0,for_sale,85000.0,NaN,NaN,21.08,1188097.0,Aguada,Puerto Rico,602.0,NaN,NaN
62,102017.0,for_sale,47000.0,NaN,NaN,0.57,1841588.0,Aguada,Puerto Rico,602.0,NaN,NaN
65,46019.0,for_sale,50000.0,NaN,NaN,0.65,1946326.0,Aguada,Puerto Rico,602.0,NaN,NaN
66,66561.0,for_sale,3250000.0,NaN,6.0,0.06,1856959.0,Rincon,Puerto Rico,602.0,NaN,NaN
67,47983.0,for_sale,39500.0,NaN,NaN,0.24,1902841.0,Aguada,Puerto Rico,602.0,NaN,NaN
68,102017.0,for_sale,249500.0,NaN,NaN,33.35,1032552.0,Aguada,Puerto Rico,602.0,NaN,NaN


Missing values investigation
A significant portion of missing values in bedrooms and house size appear to correspond to land listining or properties with incomplete property details.

These records still contains valuable information such as price, loction and lot size and therefore should not be removed from the dataset.

## 4. Property Status Analysis

This section examines the distribution of property statuses to better understand the composition of the dataset.

In [27]:
df['status'].value_counts()

status
for_sale          1389306
sold               812009
ready_to_build      25067
Name: count, dtype: int64

### Findings

The majority of records correspond to properties currently listed for sale, followed by sold properties and a smaller number of ready-to-build listings.

## 5. Outlier Detection

This section investigates extreme values in key numerical attributes such as price, house size, bedrooms and bathrooms.

Descriptive statistics

In [11]:
df.describe()

,brokered_by,price,bed,bath,acre_lot,street,zip_code,house_size
count,2.221849e+06,2.224841e+06,1.745065e+06,1.714611e+06,1.900793e+06,2.215516e+06,2.226083e+06,1.657898e+06
mean,5.293989e+04,5.241955e+05,3.275841e+00,2.496440e+00,1.522303e+01,1.012325e+06,5.218668e+04,2.714471e+03
std,3.064275e+04,2.138893e+06,1.567274e+00,1.652573e+00,7.628238e+02,5.837635e+05,2.895408e+04,8.081635e+05
min,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
25%,2.386100e+04,1.650000e+05,3.000000e+00,2.000000e+00,1.500000e-01,5.063128e+05,2.961700e+04,1.300000e+03
50%,5.288400e+04,3.250000e+05,3.000000e+00,2.000000e+00,2.600000e-01,1.012766e+06,4.838200e+04,1.760000e+03
75%,7.918300e+04,5.500000e+05,4.000000e+00,3.000000e+00,9.800000e-01,1.521173e+06,7.807000e+04,2.413000e+03
max,1.101420e+05,2.147484e+09,4.730000e+02,8.300000e+02,1.000000e+05,2.001357e+06,9.999900e+04,1.040400e+09


In [20]:
df.nlargest(10, 'bed')[['price', 'bed', 'bath', 'city', 'state']]

,price,bed,bath,city,state
583135,225000.0,473.0,157.0,Brent,Alabama
1379204,440000.0,444.0,222.0,Sublimity,Oregon
2188928,430000.0,444.0,222.0,Sublimity,Oregon
1378596,699000.0,222.0,830.0,Dallas,Oregon
1370041,1695000.0,212.0,212.0,Woodland,Washington
401858,575000.0,210.0,NaN,Cleveland,Tennessee
957227,189000.0,190.0,163.0,New Orleans,Louisiana
1382122,549000.0,148.0,74.0,Springfield,Oregon
156556,2100000.0,142.0,NaN,Rochester,New York
637798,5500000.0,136.0,NaN,Memphis,Tennessee


In [21]:
df.nlargest(10, 'house_size')[['price', 'house_size', 'bed', 'bath', 'city', 'state']]

,price,house_size,bed,bath,city,state
1213684,160000.0,1.040400e+09,NaN,NaN,Tacna,Arizona
1406699,3480000.0,1.299220e+07,NaN,NaN,Warden,Washington
434026,739000.0,9.842382e+06,NaN,NaN,Pierson,Florida
1127005,650000.0,7.971480e+06,NaN,NaN,Rocksprings,Texas
1406177,1134050.0,3.484800e+06,NaN,NaN,Wenatchee,Washington
1106095,1995000.0,3.434706e+06,NaN,NaN,Floresville,Texas
204748,289999.0,1.560780e+06,5.0,2.0,Philadelphia,Pennsylvania
1126846,153600.0,1.454468e+06,NaN,NaN,Rocksprings,Texas
2254,8250000.0,1.450112e+06,5.0,6.0,Culebra,Puerto Rico
1095437,995000.0,1.306800e+06,NaN,NaN,Castroville,Texas


In [22]:
df.nlargest(10, 'price')[['price', 'house_size', 'bed', 'bath', 'city', 'state']]

,price,house_size,bed,bath,city,state
221994,2.147484e+09,885.0,2.0,2.0,International,California
941987,1.000000e+09,NaN,NaN,NaN,Eureka,Kansas
72903,8.750000e+08,2440.0,9.0,2.0,Bronx,New York
1288464,5.150000e+08,1048.0,2.0,2.0,San Diego,California
2054444,2.950000e+08,NaN,21.0,49.0,Los Angeles,California
375790,2.815000e+08,NaN,3.0,3.0,Atlanta,Georgia
1258956,2.500000e+08,NaN,19.0,33.0,Beverly Hills,California
1051157,2.125000e+08,NaN,NaN,NaN,Kosse,Texas
105334,1.690000e+08,8255.0,6.0,9.0,New York City,New York
1258584,1.650000e+08,56500.0,14.0,27.0,Los Angeles,California


In [23]:
df['house_size'].quantile([0.50,0.75,0.90,0.95,0.99,0.999])

0.500     1760.000
0.750     2413.000
0.900     3279.000
0.950     4008.000
0.990     6439.030
0.999    14512.927
Name: house_size, dtype: float64

### House Size Outlier Analysis

The distribution of house sizes shows that:

- 99% of properties are smaller than 6,439 sqft.
- 99.9% of properties are smaller than 14,513 sqft.

However, a small number of records contain extremely large values that are several orders of magnitude above the expected range. These records are likely data quality issues, incorrect unit measurements, or non-residential properties.

These outliers will be excluded from residential market analysis while preserving the raw records in the staging layer.

In [25]:
df['price'].quantile([0.50,0.75,0.90,0.95,0.99,0.999])

0.500      325000.0
0.750      550000.0
0.900      949000.0
0.950     1495000.0
0.990     3899000.0
0.999    14500000.0
Name: price, dtype: float64

### Findings

Extreme outliers were identified in price and house size.

Most properties fall within a reasonable range, while a small number of records contain values that are significantly higher than the rest of the dataset.

These records will require further validation during the cleaning phase.

## 6. Data Quality Findings

The dataset contains more than 2.2 million property records and is generally suitable for analytical purposes.

Key observations:

- Very low percentage of missing values in location and pricing fields.
- Significant missing values in bedrooms, bathrooms and house size.
- Previous sale date is missing for approximately one-third of the records.
- Extreme outliers exist in price and house size.
- Raw data should be preserved while applying business validation rules in the analytical layer.

## 7. Next Steps

Based on the profiling results, the next phase of the project will focus on data cleaning and transformation.

The following tasks have been identified:

- Convert date fields to proper datetime format.
- Standardize column names.
- Investigate and handle missing values.
- Define business rules for outlier treatment.
- Create derived metrics such as price per square foot.
- Prepare the dataset for loading into PostgreSQL.